# 04 — Observed Legislative Advancement and Predictive Modeling


    **Research objective.** Estimate the descriptive association between bill policy orientation and observed advancement, and evaluate whether introduction-time characteristics classify advancement out of sample.

    **Inputs.** `../data/processed/bills_scored.csv`.

    **Methods.** A bill is coded as advanced when its derived status indicates passage, enactment, or veto; its passage count is positive; or a passed vote event is recorded. Wilson intervals summarize policy-orientation groups. A five-fold stratified logistic model uses only policy orientation, sponsor counts, originating chamber, introduction year, and region; a majority-class classifier provides the baseline. Random seed 149 fixes all folds.

    **Outputs.** Advancement figures, subgroup tables, model metrics, confusion matrix, and a leakage audit in `../output/`.

In [1]:
from __future__ import annotations

import json

import math

import re

from pathlib import Path

import matplotlib

import matplotlib.dates as mdates

import matplotlib.pyplot as plt

import numpy as np

import pandas as pd

from scipy.stats import spearmanr

from sklearn.compose import ColumnTransformer

from sklearn.decomposition import LatentDirichletAllocation

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

from sklearn.model_selection import (
    GroupShuffleSplit,
    StratifiedKFold,
    cross_val_predict,
    train_test_split,
)

from sklearn.pipeline import Pipeline

from sklearn.preprocessing import OneHotEncoder, StandardScaler

matplotlib.use("Agg")

SEED = 149

COLORS = {
    "restrictive": "#D55E00",
    "neutral": "#7A7A7A",
    "supportive": "#009E73",
    "blue": "#0072B2",
    "orange": "#E69F00",
    "purple": "#CC79A7",
    "sky": "#56B4E9",
}

REGIONS = {
    "California": "West", "Oregon": "West",
    "Illinois": "Midwest", "Indiana": "Midwest", "Minnesota": "Midwest",
    "North Dakota": "Midwest", "Ohio": "Midwest",
    "Georgia": "South", "Kentucky": "South", "Maryland": "South",
    "Oklahoma": "South", "Texas": "South", "Virginia": "South",
    "Maine": "Northeast", "New York": "Northeast",
}

def save_table(frame: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(path, index=False)
    print(f"Saved {len(frame):,} rows -> {path}")

def save_figure(fig: plt.Figure, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=300, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print(f"Saved figure -> {path}")

def wilson_interval(successes: int, total: int, z: float = 1.96) -> tuple[float, float]:
    if total == 0:
        return (np.nan, np.nan)
    p = successes / total
    denominator = 1 + z * z / total
    center = (p + z * z / (2 * total)) / denominator
    margin = z * math.sqrt((p * (1 - p) + z * z / (4 * total)) / total) / denominator
    return center - margin, center + margin

def load_scored_bills(path: Path) -> pd.DataFrame:
    bills = pd.read_csv(path, low_memory=False)
    for column in ["first_action_date", "latest_action_date", "latest_passage_date"]:
        bills[column] = pd.to_datetime(bills[column], errors="coerce", utc=True)
    numeric = [
        "supportiveness_score", "supportiveness_confidence", "sponsor_count",
        "primary_sponsor_count", "passage_count", "passed_vote_event_count",
    ]
    for column in numeric:
        bills[column] = pd.to_numeric(bills[column], errors="coerce")
    bills["advanced"] = (
        bills["derived_status"].isin(["passed_chamber_or_legislature", "enacted", "vetoed"])
        | bills["passage_count"].fillna(0).gt(0)
        | bills["passed_vote_event_count"].fillna(0).gt(0)
    ).astype(int)
    bills["score_numeric"] = bills["supportiveness_score"]
    bills["introduction_year"] = bills["first_action_date"].dt.year.astype("Int64")
    bills["region"] = bills["state"].map(REGIONS).fillna("Other")
    return bills

def scored_analysis_sample(bills: pd.DataFrame) -> pd.DataFrame:
    mask = (
        bills["dc_relevant"].fillna("").str.lower().eq("yes")
        & bills["supportiveness_status"].fillna("").str.lower().eq("scored")
        & bills["score_numeric"].between(1, 10)
    )
    out = bills.loc[mask].copy()
    out["supportiveness_group"] = pd.cut(
        out["score_numeric"], bins=[0, 3, 6, 10],
        labels=["Restrictive (1-3)", "Neutral or mixed (4-6)", "Supportive (7-10)"],
    )
    return out

def make_bill_model_pipeline(numeric: list[str], categorical: list[str], class_weight="balanced") -> Pipeline:
    preprocess = ColumnTransformer([
        ("numeric", Pipeline([("scale", StandardScaler())]), numeric),
        ("categorical", OneHotEncoder(handle_unknown="ignore", drop="first"), categorical),
    ])
    return Pipeline([
        ("preprocess", preprocess),
        ("model", LogisticRegression(max_iter=2000, class_weight=class_weight, random_state=SEED)),
    ])

def classification_metrics(y_true, y_pred, y_prob, model_name: str) -> dict:
    return {
        "model": model_name,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_prob) if len(np.unique(y_true)) == 2 else np.nan,
        "n": len(y_true), "positive_n": int(np.sum(y_true)),
    }

def evaluate_bill_models(data: pd.DataFrame, social_columns: list[str] | None = None) -> tuple[pd.DataFrame, np.ndarray, np.ndarray, np.ndarray]:
    numeric = ["score_numeric", "sponsor_count", "primary_sponsor_count", "introduction_year"]
    if social_columns:
        numeric += social_columns
    categorical = ["originating_chamber", "region"]
    keep = ["advanced"] + numeric + categorical
    model_data = data[keep].copy()
    for column in numeric:
        model_data[column] = pd.to_numeric(model_data[column], errors="coerce")
        model_data[column] = model_data[column].fillna(model_data[column].median()).fillna(0)
    model_data[categorical] = model_data[categorical].fillna("Missing")
    X, y = model_data[numeric + categorical], model_data.advanced.astype(int)
    folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    pipeline = make_bill_model_pipeline(numeric, categorical)
    pred = cross_val_predict(pipeline, X, y, cv=folds, method="predict")
    prob = cross_val_predict(pipeline, X, y, cv=folds, method="predict_proba")[:, 1]
    majority = int(y.mean() >= .5)
    base_pred = np.repeat(majority, len(y))
    base_prob = np.repeat(y.mean(), len(y))
    label = "Social-augmented legislative model" if social_columns else "Baseline legislative model"
    metrics = pd.DataFrame([
        classification_metrics(y, base_pred, base_prob, "Majority-class baseline"),
        classification_metrics(y, pred, prob, label),
    ])
    return metrics, y.to_numpy(), pred, prob

def run_bill_advancement(root: Path) -> dict:
    bills = scored_analysis_sample(load_scored_bills(root / "data/processed/bills_scored.csv"))
    print(f"Analysis rows: {len(bills):,}; advanced: {bills.advanced.sum():,}; not advanced: {(1-bills.advanced).sum():,}")
    print("A value of zero indicates that advancement was not observed by the collection date; it does not indicate permanent failure.")
    group_order = ["Restrictive (1-3)", "Neutral or mixed (4-6)", "Supportive (7-10)"]
    rows = []
    for group in group_order:
        x = bills.loc[bills.supportiveness_group.astype(str).eq(group), "advanced"]
        low, high = wilson_interval(int(x.sum()), len(x))
        rows.append({"supportiveness_group": group, "number_of_bills": len(x), "advanced_bills": int(x.sum()), "advancement_rate": x.mean(), "ci_low": low, "ci_high": high})
    rates = pd.DataFrame(rows)
    save_table(rates, root / "output/tables/advancement_by_supportiveness.csv")
    fig, ax = plt.subplots(figsize=(9, 6))
    colors = [COLORS["restrictive"], COLORS["neutral"], COLORS["supportive"]]
    yerr = np.vstack([rates.advancement_rate-rates.ci_low, rates.ci_high-rates.advancement_rate]) * 100
    bars = ax.bar(range(3), rates.advancement_rate*100, color=colors, yerr=yerr, capsize=5)
    for bar, row in zip(bars, rates.itertuples()):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+3.5, f"{100*row.advancement_rate:.1f}%\n(n={row.number_of_bills})", ha="center", va="bottom")
    ax.set_xticks(range(3), group_order)
    ax.set_ylabel("Bills with observed advancement (%)")
    ax.set_title(f"Observed Legislative Advancement by AI-Assisted Policy Orientation (n={len(bills):,})\nError bars report 95% Wilson confidence intervals")
    ax.set_ylim(0, max(70, (rates.ci_high.max()*100)+12))
    ax.grid(axis="y", alpha=.2)
    fig.tight_layout()
    save_figure(fig, root / "output/figures/advancement_by_supportiveness.png")

    by_state = bills.groupby("state").agg(n=("bill_id","nunique"), advanced=("advanced","sum"), advancement_rate=("advanced","mean")).reset_index()
    by_year = bills.groupby("introduction_year", dropna=False).agg(n=("bill_id","nunique"), advanced=("advanced","sum"), advancement_rate=("advanced","mean")).reset_index()
    save_table(by_state, root / "output/tables/advancement_by_state.csv")
    save_table(by_year, root / "output/tables/advancement_by_year.csv")

    fig, ax = plt.subplots(figsize=(9, 5.5))
    advanced_scores = [bills.loc[bills.advanced.eq(v), "score_numeric"] for v in [0,1]]
    ax.hist(advanced_scores, bins=np.arange(.5,10.6,1), stacked=False, density=True, alpha=.65,
            label=[f"No advancement observed (n={len(advanced_scores[0])})", f"Advanced (n={len(advanced_scores[1])})"],
            color=[COLORS["neutral"], COLORS["blue"]])
    ax.axvline(5, color="black", ls="--", lw=1, label="Neutral midpoint")
    ax.set_xlabel("AI-assisted policy-orientation score")
    ax.set_ylabel("Within-group density")
    ax.set_title("Distribution of Policy-Orientation Scores by Observed Advancement")
    ax.legend()
    fig.tight_layout()
    save_figure(fig, root / "output/figures/supportiveness_by_advancement.png")

    metrics, y, pred, prob = evaluate_bill_models(bills)
    save_table(metrics, root / "output/tables/baseline_advancement_model_metrics.csv")
    cm = confusion_matrix(y, pred)
    save_table(pd.DataFrame(cm, index=["actual_0","actual_1"], columns=["predicted_0","predicted_1"]).reset_index(), root / "output/tables/baseline_advancement_confusion_matrix.csv")
    leakage = pd.DataFrame({
        "field": ["supportiveness_score","sponsor_count","primary_sponsor_count","originating_chamber","introduction_year","region","passage_count","passed_vote_event_count","derived_status","latest_action_description","latest_passage_date"],
        "used": [True,True,True,True,True,True,False,False,False,False,False],
        "timing": ["coded bill content","near introduction","near introduction","at introduction","at introduction","fixed control","post-outcome","post-outcome","post-outcome","post-outcome","post-outcome"],
    })
    save_table(leakage, root / "output/tables/baseline_model_leakage_audit.csv")
    return {"n": len(bills), "advanced": int(bills.advanced.sum()), "metrics": metrics.to_dict("records")}

ROOT = Path("..")
print("Random seed:", SEED)

Random seed: 149


In [2]:
results = run_bill_advancement(ROOT)
results

Analysis rows: 395; advanced: 129; not advanced: 266
A value of zero indicates that advancement was not observed by the collection date; it does not indicate permanent failure.
Saved 3 rows -> ../output/tables/advancement_by_supportiveness.csv


Saved figure -> ../output/figures/advancement_by_supportiveness.png
Saved 15 rows -> ../output/tables/advancement_by_state.csv
Saved 5 rows -> ../output/tables/advancement_by_year.csv


Saved figure -> ../output/figures/supportiveness_by_advancement.png


Saved 2 rows -> ../output/tables/baseline_advancement_model_metrics.csv
Saved 2 rows -> ../output/tables/baseline_advancement_confusion_matrix.csv
Saved 11 rows -> ../output/tables/baseline_model_leakage_audit.csv


{'n': 395,
 'advanced': 129,
 'metrics': [{'model': 'Majority-class baseline',
   'accuracy': 0.6734177215189874,
   'precision': 0.0,
   'recall': 0.0,
   'f1': 0.0,
   'roc_auc': 0.5,
   'n': 395,
   'positive_n': 129},
  {'model': 'Baseline legislative model',
   'accuracy': 0.6911392405063291,
   'precision': 0.517948717948718,
   'recall': 0.7829457364341085,
   'f1': 0.6234567901234568,
   'roc_auc': 0.7648335956169493,
   'n': 395,
   'positive_n': 129}]}

## Interpretation, inferential scope, and limitations

A zero means “no advancement observed by collection time,” not permanent failure. The comparison is associational: score, institutional differences, sponsorship, topic, and timing can all be related. Cross-validation measures performance within this collected sample, not a future causal effect or guaranteed performance in new legislative sessions.